# 📦 Dense Embeddings (Traditional RAG)
## Create single-vector embeddings using SentenceTransformers

This notebook implements the **traditional RAG approach** where each document becomes a single dense vector. We'll use the `all-MiniLM-L6-v2` model for optimal speed/performance balance.

### 🎯 What we'll accomplish:
1. Load the SentenceTransformer model
2. Create 384-dimensional embeddings for all restaurant reviews
3. Store embeddings in LanceDB
4. Implement basic dense search functionality
5. Analyze the limitations of single-vector approach

In [ ]:
# 1. Setup & Imports
import sys
sys.path.append('../..')  # Add project root to path
from setup import *
import time
from tqdm import tqdm

print("✅ Setup imported successfully!")
print(f"📂 Working directory: {os.getcwd()}")
print(f"🎯 Dense model: {os.getenv('DENSE_MODEL_NAME')}")
print(f"📏 Embedding dimensions: {os.getenv('EMBEDDING_DIMENSION', '384')}")
print(f"🔧 Device: {get_device()}")

## 🤖 Load Dense Embedding Model

We're using **all-MiniLM-L6-v2** based on 2025 research:
- **384 dimensions** - Perfect balance of quality and speed
- **5x faster** than larger models with only ~3% accuracy drop
- **22M parameters** - Efficient for real-time applications
- **Top choice for RAG** - Validated by MTEB benchmarks

In [ ]:
from IPython.display import Image, display

def show_image(image_filename="ColBERT/explainibility.png", notebook_dir_env="NOTEBOOKS_DIR"):
    """
    Displays an image from the ColBERT explainability directory.
    
    Args:
        image_filename (str): Relative path to the image file within the notebook directory.
        notebook_dir_env (str): Name of the environment variable containing the notebook directory path.
    """
    import os
    notebook_dir = os.getenv(notebook_dir_env)
    if notebook_dir is None:
        raise ValueError(f"Environment variable '{notebook_dir_env}' is not set.")
    image_path = os.path.join(notebook_dir, image_filename)
    print("Image path:", image_path)
    display(Image(filename=image_path))

In [ ]:
# 2. Load Dense Model
from sentence_transformers import SentenceTransformer
import torch

print("🔄 Loading dense embedding model...")

# Use CPU for M1 compatibility (research shows minimal performance impact)
device_for_dense = 'cpu' if get_device() == 'mps' else get_device()
print(f"📱 Using device: {device_for_dense}")

# Load the research-validated model
model_name = os.getenv('DENSE_MODEL_NAME')
dense_model = SentenceTransformer(model_name, device=device_for_dense)

print(f"✅ Model loaded: {model_name}")
print(f"📏 Output dimensions: {dense_model.get_sentence_embedding_dimension()}")
print(f"🔧 Running on: {device_for_dense}")

# Test with sample embedding
test_text = "Italian restaurant with outdoor seating and great pasta"
test_start = time.time()
test_embedding = dense_model.encode(test_text)
test_time = time.time() - test_start

print(f"\n🧪 Test embedding:")
print(f"   Input: '{test_text}'")
print(f"   Output shape: {test_embedding.shape}")
print(f"   Encoding time: {test_time:.3f} seconds")
print(f"   First 5 values: {test_embedding[:5]}")

In [ ]:
show_image("Colbert/advanced_rag_concepts.png")

## Explanation for Dense Vectors

In [ ]:
# Credit: Antoin Chaffin, Lighton
show_image("Colbert/dense.png")

In [ ]:
# Credits: https://weaviate.io/blog/late-interaction-overview
show_image("Colbert/dense2.png")

In [ ]:
# Credits: https://weaviate.io/blog/late-interaction-overview
show_image("Colbert/full_interaction.png")

In [ ]:
show_image("Colbert/selective.png")

## 📊 Process Restaurant Data

Load our restaurant reviews and prepare them for embedding. We'll combine restaurant name and review text to give the model full context.

In [ ]:
# 3. Process Restaurant Data
print("📊 Loading restaurant reviews...")

# Load the data
data_path = os.getenv('RESTAURANT_REVIEWS_CSV')
df = pd.read_csv(data_path)

print(f"✅ Loaded {len(df)} restaurant reviews")
print(f"📋 Columns: {list(df.columns)}")


# Prepare texts for embedding (combine restaurant name + review)
texts_for_embedding = []
for _, row in df.iterrows():
    # Combine restaurant name and review for better context
    combined_text = f"{row['restaurant']}: {row['review']}"
    texts_for_embedding.append(combined_text)

print(f"\n📝 Sample combined text:")
print(f"'{texts_for_embedding[0][:100]}...'")

# Show text statistics
text_lengths = [len(text) for text in texts_for_embedding]
print(f"\n📈 Text Statistics:")
print(f"   Average length: {np.mean(text_lengths):.0f} characters")
print(f"   Range: {min(text_lengths)} - {max(text_lengths)} characters")
print(f"   Total texts to embed: {len(texts_for_embedding)}")

## ⚡ Create Dense Embeddings

Now we'll create **single dense vectors** for each restaurant review. This is the core of traditional RAG - each document becomes one 384-dimensional point in vector space.

**Key limitation**: All the rich information in the review gets compressed into just 384 numbers!

In [ ]:
# 4. Create Dense Embeddings
print("⚡ Creating dense embeddings for all reviews...")
print("📦 Each review → Single 384-dimensional vector")

start_time = time.time()

# Create embeddings with progress bar
dense_embeddings = []
for i, text in enumerate(tqdm(texts_for_embedding, desc="Creating embeddings")):
    embedding = dense_model.encode(text)
    dense_embeddings.append(embedding)
    
    # Show progress for first few
    if i < 3:
        restaurant_name = df.iloc[i]['restaurant']
        print(f"  ✅ {i+1:2d}. {restaurant_name:20} → {embedding.shape} vector")

embedding_time = time.time() - start_time

print(f"\n🎉 Dense embeddings created successfully!")
print(f"   📊 Total embeddings: {len(dense_embeddings)}")
print(f"   📏 Shape per embedding: {dense_embeddings[0].shape}")
print(f"   ⏱️  Total time: {embedding_time:.2f} seconds")
print(f"   🚀 Average per review: {embedding_time/len(dense_embeddings):.3f} seconds")

# Convert to numpy array for easier manipulation
dense_embeddings_array = np.array(dense_embeddings)
print(f"   🔢 Final array shape: {dense_embeddings_array.shape} (reviews × dimensions)")

## 🗄️ Store Embeddings in LanceDB

Store our dense embeddings in LanceDB for fast similarity search. Each document gets stored with its metadata and single vector.

In [ ]:
# 5. Store in LanceDB
import lancedb
import pyarrow as pa

print("🗄️ Storing dense embeddings in LanceDB...")

# Connect to database
db_path = os.getenv('VECTOR_STORE_DIR')
db = lancedb.connect(db_path)
print(f"📍 Database path: {db_path}")

# Prepare data for LanceDB
dense_data = []
for idx, (_, row) in enumerate(df.iterrows()):
    doc = {
        'id': int(row['id']),
        'restaurant': row['restaurant'],
        'review': row['review'],
        'reviewer': row['reviewer'],
        'rating': int(row['rating']),
        'text': texts_for_embedding[idx],  # Combined text
        'dense_embedding': dense_embeddings[idx].tolist()  # Convert numpy to list
    }
    dense_data.append(doc)

# Define schema
embedding_dim = int(os.getenv('EMBEDDING_DIMENSION', '384'))
dense_schema = pa.schema([
    pa.field("id", pa.int64()),
    pa.field("restaurant", pa.string()),
    pa.field("review", pa.string()),
    pa.field("reviewer", pa.string()),
    pa.field("rating", pa.int64()),
    pa.field("text", pa.string()),
    pa.field("dense_embedding", pa.list_(pa.float32(), embedding_dim))
])

# Create table (drop existing if present)
table_name = "dense_reviews"
if table_name in db.table_names():
    db.drop_table(table_name)
    print(f"🗑️  Dropped existing {table_name} table")

dense_table = db.create_table(table_name, dense_data, schema=dense_schema)

print(f"✅ Created {table_name} table")
print(f"   📊 Records: {len(dense_table)}")
print(f"   📋 Schema: {[field.name for field in dense_table.schema]}")
print(f"   💾 Storage: Single vector per document")

# Verify storage
sample_record = dense_table.to_pandas().iloc[0]
print(f"\n🔍 Sample stored record:")
print(f"   Restaurant: {sample_record['restaurant']}")
print(f"   Rating: {'⭐' * sample_record['rating']}")
print(f"   Embedding shape: {len(sample_record['dense_embedding'])} dimensions")

## 🔍 Basic Dense Search

Implement traditional RAG search using cosine similarity between query and document vectors. This is the **baseline** we'll compare ColBERT against.

In [ ]:
# 6. Basic Dense Search


def search_dense_reviews(query, top_k=3):
    """
    Search restaurant reviews using dense embeddings (traditional RAG)
    
    Args:
        query (str): Search query
        top_k (int): Number of results to return
    
    Returns:
        pandas.DataFrame: Top matching reviews with similarity scores
    """
    print(f"🔍 Dense search query: '{query}'")
    
    # Encode query with the same model
    query_start = time.time()
    query_embedding = dense_model.encode(query)
    query_time = time.time() - query_start
    
    print(f"   ⚡ Query encoding time: {query_time:.3f} seconds")
    
    # Search using LanceDB
    search_start = time.time()
    results = dense_table.search(query_embedding).limit(top_k).to_pandas()
    search_time = time.time() - search_start
    
    print(f"   🚀 Search time: {search_time:.3f} seconds")
    print(f"   📊 Found {len(results)} results")
    
    return results

print("✅ Dense search function ready!")
print("📦 How it works:")
print("   1. Query → Dense model → Single 384-dim vector")
print("   2. Compare query vector vs all document vectors")
print("   3. Return top-K most similar documents")
print("   4. Uses cosine similarity in 384-dimensional space")

In [ ]:
# Test dense search with sample queries
test_queries = [
    "Italian restaurant",
    "quiet place for working with laptop", 
    "expensive restaurant",
    "family friendly restaurant"
]

print("🧪 Testing dense search with sample queries:")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
    print(f"\n🔍 Test {i}: {query}")
    print("-" * 40)
    
    try:
        results = search_dense_reviews(query, top_k=2)
        
        for idx, row in results.iterrows():
            distance = row.get('_distance', 0)
            similarity = 1 - distance  # Convert distance to similarity
            
            print(f"   📍 #{row['id']} - {row['restaurant']}")
            print(f"      ⭐ {row['rating']}/5") #| 📊 Similarity: {similarity:.3f}")
            print(f"      💭 {row['review'][:]}...")
            
    except Exception as e:
        print(f"   ❌ Search failed: {e}")
        
    if i < len(test_queries):
        print()

print("\n🎉 Dense search testing complete!")

## ⚠️ Dense Embedding Limitations

Now let's discuss the key limitations of dense embeddings that ColBERT addresses.

In [ ]:
# 8. Limitations Analysis
print("⚠️  DENSE EMBEDDING LIMITATIONS")
print("=" * 50)

print("\n🗜️  1. INFORMATION COMPRESSION:")
print(f"   • Rich review text → Just 384 numbers")
print(f"   • Complex concepts squeezed into single point")
print(f"   • Relationships between concepts lost")

print("\n🔍 2. SEARCH LIMITATIONS:")
print("   • Query: 'Italian budget-friendly outdoor' → Single vector")
print("   • Can't match individual concepts separately")
print("   • Struggles with multi-constraint queries")
print("   • Poor handling of contradictory terms")

print("\n📊 3. MATCHING PROCESS:")
print("   • Only overall similarity score available")
print("   • No insight into WHICH parts matched")
print("   • Black box similarity calculation")
print("   • Hard to debug poor results")


print("\n🎯 WHY COLBERT WILL BE BETTER:")
print("   ✅ Preserves individual token information")
print("   ✅ Can match 'Italian' separately from 'outdoor'")
print("   ✅ Handles multi-constraint queries naturally")
print("   ✅ Provides interpretable token-level matching")

print("\n🏁 DENSE EMBEDDING SUMMARY:")
print("   📦 Fast and simple to implement")
print("   📊 Good for basic similarity search")
print("   ⚠️  Limited for complex, multi-faceted queries")
print("   🎯 Perfect baseline for comparison")

print("\n🚀 NEXT: ColBERT will show token-level precision!")